# Data Preprocessing

In [1]:
import pandas as pd
import numpy as np

# Load dataset
data = pd.read_csv('owid_covid_data.csv')

# Convert the date column to datetime for easier manipulation
data['date'] = pd.to_datetime(data['date'])

# Sort data by date
data.sort_values(by='date', inplace=True)

# Handle Outliers 

# Function to calculate IQR and remove outliers
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)  # First quartile
    Q3 = df[column].quantile(0.75)  # Third quartile
    IQR = Q3 - Q1  # Interquartile range
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

# Remove outliers from hospitalization_rate and icu_rate
data = remove_outliers(data, 'hosp_patients_per_million')
data = remove_outliers(data, 'icu_patients_per_million')

# Assign Risk Levels 

# Percentile-based dynamic thresholds for hospitalization_rate and icu_rate
h_rate_thresholds = data['hosp_patients_per_million'].quantile([0.25, 0.5, 0.75]).values
icu_rate_thresholds = data['icu_patients_per_million'].quantile([0.25, 0.5, 0.75]).values

# Dynamic risk assignment function
def assign_dynamic_risk(hospitalization_rate, icu_rate):
    if hospitalization_rate > h_rate_thresholds[2] or icu_rate > icu_rate_thresholds[2]:
        return 'High'
    elif hospitalization_rate > h_rate_thresholds[1] or icu_rate > icu_rate_thresholds[1]:
        return 'Medium'
    else:
        return 'Low'

# Apply the function to create a new 'risk_level' column
data['risk_level'] = data.apply(
    lambda x: assign_dynamic_risk(x['hosp_patients_per_million'], x['icu_patients_per_million']), axis=1
)

# Validation of Risk Levels 

# Summary of risk levels
risk_summary = data['risk_level'].value_counts()
print("Risk Level Summary:\n", risk_summary)

# Save Processed Data

# Save the preprocessed data to a new CSV file
data.to_csv('preprocessed_owid_covid_data.csv', index=False)

# Preview the processed dataset
print(data[['date', 'hosp_patients_per_million', 'icu_patients_per_million', 'risk_level']].head())

Risk Level Summary:
 Low       10428
High      10423
Medium     7507
Name: risk_level, dtype: int64
             date  hosp_patients_per_million  icu_patients_per_million  \
185325 2020-02-24                      2.151                     0.440   
185326 2020-02-25                      2.541                     0.593   
185327 2020-02-26                      2.778                     0.610   
185328 2020-02-27                      5.149                     0.949   
185329 2020-02-28                      6.928                     1.084   

       risk_level  
185325        Low  
185326        Low  
185327        Low  
185328        Low  
185329        Low  
